# 🚀 Ultimate RAG Pipelines Cheatsheet

A comprehensive guide to building Retrieval-Augmented Generation (RAG) systems with ChromaDB.

## Table of Contents
1. **Setup & Installation**
2. **Simple RAG Pipeline**
3. **RAG with Advanced Chunking Strategies**
4. **RAG with Reranking**
5. **Hybrid Search RAG (Dense + Sparse)**
6. **Multi-Query RAG**
7. **Self-Query RAG**
8. **Parent Document Retriever**
9. **Multi-Agent RAG System**
10. **Evaluation & Metrics**

---

## 1. 📦 Setup & Installation

In [ ]:
# Install required packages
# !pip install chromadb langchain langchain-openai langchain-community langchain-huggingface
# !pip install sentence-transformers tiktoken rank_bm25 cohere
# !pip install openai anthropic google-generativeai
# !pip install faiss-cpu pypdf unstructured

# Core imports
import os
import json
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

# ChromaDB
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions

# LangChain Core
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter,
    MarkdownHeaderTextSplitter,
    HTMLHeaderTextSplitter,
)
from langchain.schema import Document
from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    DirectoryLoader,
    UnstructuredMarkdownLoader,
)

# Embeddings
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.embeddings import CohereEmbeddings

# Vector Stores
from langchain_community.vectorstores import Chroma

# LLMs
from langchain_openai import ChatOpenAI
from langchain_community.llms import Ollama

# Chains and Retrievers
from langchain.chains import RetrievalQA, ConversationalRetrievalChain
from langchain.chains.query_constructor.base import AttributeInfo
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

# Prompts
from langchain.prompts import PromptTemplate, ChatPromptTemplate

print("✅ All imports loaded successfully!")

In [ ]:
# ============================================
# 🔑 API Keys Configuration
# ============================================

# Set your API keys (use environment variables in production!)
os.environ["OPENAI_API_KEY"] = "your-openai-api-key"  # Replace with your key
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"  # For reranking
os.environ["ANTHROPIC_API_KEY"] = "your-anthropic-api-key"  # Optional

# Or load from .env file
# from dotenv import load_dotenv
# load_dotenv()

print("🔑 API keys configured!")

In [ ]:
# ============================================
# 📚 Sample Documents for Testing
# ============================================

sample_documents = [
    Document(
        page_content="""
        Machine Learning is a subset of artificial intelligence that enables systems 
        to learn and improve from experience without being explicitly programmed. 
        It focuses on developing algorithms that can access data and use it to learn 
        for themselves. The process begins with observations or data, such as examples, 
        direct experience, or instruction.
        """,
        metadata={"source": "ml_basics.txt", "topic": "machine_learning", "chapter": 1}
    ),
    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        Neural networks are inspired by the human brain and consist of layers of interconnected
        nodes. Deep learning excels at pattern recognition, image classification, natural 
        language processing, and speech recognition tasks.
        """,
        metadata={"source": "dl_intro.txt", "topic": "deep_learning", "chapter": 2}
    ),
    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand,
        interpret and manipulate human language. NLP draws from many disciplines including
        linguistics and computer science. Applications include sentiment analysis, machine
        translation, chatbots, and text summarization.
        """,
        metadata={"source": "nlp_guide.txt", "topic": "nlp", "chapter": 3}
    ),
    Document(
        page_content="""
        RAG (Retrieval-Augmented Generation) combines retrieval and generation for better
        responses. It first retrieves relevant documents from a knowledge base, then uses
        an LLM to generate answers based on the retrieved context. This approach reduces
        hallucinations and provides more accurate, grounded responses.
        """,
        metadata={"source": "rag_overview.txt", "topic": "rag", "chapter": 4}
    ),
    Document(
        page_content="""
        Vector databases store data as high-dimensional vectors, enabling semantic search
        capabilities. Popular vector databases include ChromaDB, Pinecone, Weaviate, and
        Milvus. They use approximate nearest neighbor (ANN) algorithms for efficient
        similarity search across millions of vectors.
        """,
        metadata={"source": "vectordb_guide.txt", "topic": "vector_databases", "chapter": 5}
    ),
    Document(
        page_content="""
        Transformer architecture revolutionized NLP with its attention mechanism.
        Introduced in 2017's "Attention is All You Need" paper, transformers process
        sequences in parallel rather than sequentially. Models like BERT, GPT, and
        T5 are all based on the transformer architecture.
        """,
        metadata={"source": "transformers.txt", "topic": "transformers", "chapter": 6}
    ),
    Document(
        page_content="""
        Fine-tuning is the process of taking a pre-trained model and training it further
        on a specific dataset for a particular task. This transfer learning approach
        allows leveraging knowledge from large datasets while adapting to specific
        use cases with limited data.
        """,
        metadata={"source": "finetuning.txt", "topic": "fine_tuning", "chapter": 7}
    ),
    Document(
        page_content="""
        Prompt engineering is the art of crafting effective prompts for LLMs.
        Good prompts are clear, specific, and provide relevant context. Techniques
        include few-shot learning, chain-of-thought prompting, and role-playing.
        Well-designed prompts can significantly improve model outputs.
        """,
        metadata={"source": "prompts.txt", "topic": "prompt_engineering", "chapter": 8}
    ),
]

print(f"📚 Loaded {len(sample_documents)} sample documents")

---
## 2. 🎯 Simple RAG Pipeline

The most basic RAG implementation: Embed documents → Store in ChromaDB → Retrieve → Generate

In [ ]:
# ============================================
# 🎯 SIMPLE RAG PIPELINE
# ============================================

class SimpleRAG:
    """
    Basic RAG pipeline with ChromaDB
    
    Flow: Documents → Embed → Store → Query → Retrieve → Generate
    """
    
    def __init__(
        self,
        collection_name: str = "simple_rag",
        embedding_model: str = "all-MiniLM-L6-v2",
        persist_directory: str = "./chroma_db"
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        
        # Initialize ChromaDB client
        self.client = chromadb.PersistentClient(path=persist_directory)
        
        # Initialize embedding function
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name=embedding_model
        )
        
        # Get or create collection
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            embedding_function=self.embedding_function,
            metadata={"hnsw:space": "cosine"}  # Use cosine similarity
        )
        
        print(f"✅ SimpleRAG initialized with collection: {collection_name}")
    
    def add_documents(self, documents: List[Document]) -> None:
        """Add documents to the collection"""
        texts = [doc.page_content for doc in documents]
        metadatas = [doc.metadata for doc in documents]
        ids = [f"doc_{i}" for i in range(len(documents))]
        
        self.collection.add(
            documents=texts,
            metadatas=metadatas,
            ids=ids
        )
        print(f"📥 Added {len(documents)} documents to collection")
    
    def query(self, query_text: str, n_results: int = 3) -> Dict[str, Any]:
        """Query the collection and return relevant documents"""
        results = self.collection.query(
            query_texts=[query_text],
            n_results=n_results,
            include=["documents", "metadatas", "distances"]
        )
        return results
    
    def generate_response(
        self,
        query: str,
        context_docs: List[str],
        llm_client=None
    ) -> str:
        """Generate response using retrieved context"""
        context = "\n\n".join(context_docs)
        
        prompt = f"""You are a helpful assistant. Use the following context to answer the question.
If you don't know the answer based on the context, say "I don't have enough information."

Context:
{context}

Question: {query}

Answer:"""
        
        # If no LLM client provided, return the prompt (for testing)
        if llm_client is None:
            return f"[PROMPT FOR LLM]:\n{prompt}"
        
        # Use OpenAI
        response = llm_client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content
    
    def rag_query(self, query: str, n_results: int = 3, llm_client=None) -> Dict[str, Any]:
        """Complete RAG pipeline: retrieve + generate"""
        # Retrieve
        results = self.query(query, n_results)
        context_docs = results["documents"][0]
        
        # Generate
        response = self.generate_response(query, context_docs, llm_client)
        
        return {
            "query": query,
            "retrieved_docs": context_docs,
            "distances": results["distances"][0],
            "response": response
        }

# Example usage
print("\n📝 Simple RAG Example:")
print("-" * 50)

In [ ]:
# Test Simple RAG (using in-memory for demo)
simple_rag = SimpleRAG(collection_name="demo_simple", persist_directory="./demo_chroma")
simple_rag.add_documents(sample_documents)

# Query example
result = simple_rag.rag_query("What is RAG and how does it work?")
print(f"Query: {result['query']}")
print(f"\n📄 Retrieved {len(result['retrieved_docs'])} documents")
for i, (doc, dist) in enumerate(zip(result['retrieved_docs'], result['distances'])):
    print(f"\n  [{i+1}] Distance: {dist:.4f}")
    print(f"      {doc[:100]}...")

---
## 3. ✂️ RAG with Advanced Chunking Strategies

Different chunking strategies for different document types:
- **Fixed-size chunking** - Split by character/token count
- **Recursive chunking** - Split by multiple separators
- **Semantic chunking** - Split by meaning/similarity
- **Document-aware chunking** - Markdown, HTML headers

In [ ]:
# ============================================
# ✂️ CHUNKING STRATEGIES
# ============================================

class ChunkingStrategies:
    """
    Various text chunking strategies for RAG pipelines
    """
    
    @staticmethod
    def fixed_size_chunking(
        text: str,
        chunk_size: int = 500,
        chunk_overlap: int = 50
    ) -> List[str]:
        """
        Simple fixed-size character chunking
        
        Best for: Uniform documents, simple use cases
        """
        splitter = CharacterTextSplitter(
            separator="\n",
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            length_function=len
        )
        return splitter.split_text(text)
    
    @staticmethod
    def recursive_chunking(
        text: str,
        chunk_size: int = 500,
        chunk_overlap: int = 50,
        separators: List[str] = None
    ) -> List[str]:
        """
        Recursive splitting with multiple separators
        
        Best for: General-purpose text, mixed content
        Tries to split by paragraph, then sentence, then word
        """
        if separators is None:
            separators = ["\n\n", "\n", ". ", " ", ""]
        
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=separators,
            length_function=len
        )
        return splitter.split_text(text)
    
    @staticmethod
    def token_chunking(
        text: str,
        chunk_size: int = 256,
        chunk_overlap: int = 20,
        encoding_name: str = "cl100k_base"
    ) -> List[str]:
        """
        Token-based chunking (uses tiktoken)
        
        Best for: LLM-optimized chunks, precise token limits
        """
        splitter = TokenTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            encoding_name=encoding_name
        )
        return splitter.split_text(text)
    
    @staticmethod
    def markdown_chunking(text: str) -> List[Document]:
        """
        Split markdown by headers
        
        Best for: Documentation, markdown files
        """
        headers_to_split_on = [
            ("#", "Header 1"),
            ("##", "Header 2"),
            ("###", "Header 3"),
        ]
        splitter = MarkdownHeaderTextSplitter(
            headers_to_split_on=headers_to_split_on
        )
        return splitter.split_text(text)
    
    @staticmethod
    def semantic_chunking(
        text: str,
        embedding_model: str = "all-MiniLM-L6-v2",
        similarity_threshold: float = 0.5,
        max_chunk_size: int = 1000
    ) -> List[str]:
        """
        Semantic chunking based on embedding similarity
        
        Best for: Preserving semantic meaning, coherent chunks
        """
        from sentence_transformers import SentenceTransformer
        import numpy as np
        
        # Split into sentences first
        sentences = text.replace('\n', ' ').split('. ')
        sentences = [s.strip() + '.' for s in sentences if s.strip()]
        
        if len(sentences) <= 1:
            return [text]
        
        # Get embeddings
        model = SentenceTransformer(embedding_model)
        embeddings = model.encode(sentences)
        
        # Group by similarity
        chunks = []
        current_chunk = [sentences[0]]
        current_embedding = embeddings[0]
        
        for i in range(1, len(sentences)):
            # Calculate similarity with current chunk
            similarity = np.dot(current_embedding, embeddings[i]) / (
                np.linalg.norm(current_embedding) * np.linalg.norm(embeddings[i])
            )
            
            current_text = ' '.join(current_chunk)
            
            if similarity > similarity_threshold and len(current_text) < max_chunk_size:
                current_chunk.append(sentences[i])
                # Update embedding as average
                current_embedding = np.mean(
                    [current_embedding, embeddings[i]], axis=0
                )
            else:
                chunks.append(' '.join(current_chunk))
                current_chunk = [sentences[i]]
                current_embedding = embeddings[i]
        
        # Don't forget the last chunk
        if current_chunk:
            chunks.append(' '.join(current_chunk))
        
        return chunks

# Demonstrate chunking strategies
demo_text = """
Machine learning is transforming industries worldwide. It enables computers to learn from data 
and make predictions without explicit programming.

Deep learning, a subset of machine learning, uses neural networks with multiple layers. 
These networks can automatically learn representations from data.

Natural Language Processing (NLP) is another exciting field. It helps computers understand 
and generate human language. Applications include chatbots, translation, and sentiment analysis.

Vector databases are essential for modern AI applications. They store embeddings and enable 
fast similarity search. Popular options include ChromaDB, Pinecone, and Weaviate.
"""

chunker = ChunkingStrategies()
print("📊 Chunking Strategies Comparison:")
print("=" * 60)

# Fixed size
fixed_chunks = chunker.fixed_size_chunking(demo_text, chunk_size=200, chunk_overlap=20)
print(f"\n1️⃣ Fixed Size Chunking: {len(fixed_chunks)} chunks")
for i, chunk in enumerate(fixed_chunks[:2]):
    print(f"   Chunk {i+1}: {chunk[:80]}...")

# Recursive
recursive_chunks = chunker.recursive_chunking(demo_text, chunk_size=200, chunk_overlap=20)
print(f"\n2️⃣ Recursive Chunking: {len(recursive_chunks)} chunks")
for i, chunk in enumerate(recursive_chunks[:2]):
    print(f"   Chunk {i+1}: {chunk[:80]}...")

In [ ]:
# ============================================
# ✂️ RAG WITH ADVANCED CHUNKING
# ============================================

class ChunkedRAG:
    """
    RAG pipeline with configurable chunking strategies
    """
    
    def __init__(
        self,
        collection_name: str = "chunked_rag",
        chunking_strategy: str = "recursive",
        chunk_size: int = 500,
        chunk_overlap: int = 50,
        persist_directory: str = "./chroma_chunked"
    ):
        self.collection_name = collection_name
        self.chunking_strategy = chunking_strategy
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.chunker = ChunkingStrategies()
        
        # Initialize ChromaDB
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name="all-MiniLM-L6-v2"
        )
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            embedding_function=self.embedding_function
        )
        
        print(f"✅ ChunkedRAG initialized with {chunking_strategy} chunking")
    
    def _chunk_text(self, text: str) -> List[str]:
        """Apply the configured chunking strategy"""
        if self.chunking_strategy == "fixed":
            return self.chunker.fixed_size_chunking(
                text, self.chunk_size, self.chunk_overlap
            )
        elif self.chunking_strategy == "recursive":
            return self.chunker.recursive_chunking(
                text, self.chunk_size, self.chunk_overlap
            )
        elif self.chunking_strategy == "token":
            return self.chunker.token_chunking(
                text, self.chunk_size // 4, self.chunk_overlap // 4
            )
        elif self.chunking_strategy == "semantic":
            return self.chunker.semantic_chunking(text)
        else:
            raise ValueError(f"Unknown chunking strategy: {self.chunking_strategy}")
    
    def add_documents(self, documents: List[Document]) -> None:
        """Add documents with chunking"""
        all_chunks = []
        all_metadatas = []
        
        for doc in documents:
            chunks = self._chunk_text(doc.page_content)
            for i, chunk in enumerate(chunks):
                all_chunks.append(chunk)
                metadata = doc.metadata.copy()
                metadata["chunk_index"] = i
                metadata["total_chunks"] = len(chunks)
                all_metadatas.append(metadata)
        
        ids = [f"chunk_{i}" for i in range(len(all_chunks))]
        
        self.collection.add(
            documents=all_chunks,
            metadatas=all_metadatas,
            ids=ids
        )
        print(f"📥 Added {len(documents)} documents as {len(all_chunks)} chunks")
    
    def query(self, query_text: str, n_results: int = 5) -> Dict[str, Any]:
        """Query with chunk-aware retrieval"""
        results = self.collection.query(
            query_texts=[query_text],
            n_results=n_results,
            include=["documents", "metadatas", "distances"]
        )
        return results

# Example: Compare different chunking strategies
print("\n📊 Comparing Chunking Strategies for RAG:")
print("=" * 60)

strategies = ["fixed", "recursive"]
for strategy in strategies:
    rag = ChunkedRAG(
        collection_name=f"demo_{strategy}",
        chunking_strategy=strategy,
        chunk_size=300,
        persist_directory=f"./demo_{strategy}"
    )
    rag.add_documents(sample_documents[:3])
    
    results = rag.query("What is deep learning?", n_results=2)
    print(f"\n🔍 Strategy: {strategy.upper()}")
    print(f"   Found {len(results['documents'][0])} relevant chunks")

---
## 4. 🎖️ RAG with Reranking

Reranking improves retrieval quality by re-scoring retrieved documents using a more sophisticated model.

**Two-stage retrieval:**
1. **First stage:** Fast retrieval (vector similarity)
2. **Second stage:** Precise reranking (cross-encoder)

In [ ]:
# ============================================
# 🎖️ RAG WITH RERANKING
# ============================================

class RerankerRAG:
    """
    RAG pipeline with two-stage retrieval:
    1. Vector search (fast, recall-oriented)
    2. Cross-encoder reranking (precise, accuracy-oriented)
    """
    
    def __init__(
        self,
        collection_name: str = "reranker_rag",
        embedding_model: str = "all-MiniLM-L6-v2",
        reranker_model: str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
        persist_directory: str = "./chroma_rerank"
    ):
        self.collection_name = collection_name
        
        # Initialize ChromaDB
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name=embedding_model
        )
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            embedding_function=self.embedding_function
        )
        
        # Initialize reranker (Cross-Encoder)
        from sentence_transformers import CrossEncoder
        self.reranker = CrossEncoder(reranker_model)
        
        print(f"✅ RerankerRAG initialized with {reranker_model}")
    
    def add_documents(self, documents: List[Document]) -> None:
        """Add documents to the collection"""
        texts = [doc.page_content for doc in documents]
        metadatas = [doc.metadata for doc in documents]
        ids = [f"doc_{i}" for i in range(len(documents))]
        
        self.collection.add(
            documents=texts,
            metadatas=metadatas,
            ids=ids
        )
        print(f"📥 Added {len(documents)} documents")
    
    def retrieve(
        self,
        query: str,
        initial_k: int = 10,
        final_k: int = 3
    ) -> List[Dict[str, Any]]:
        """
        Two-stage retrieval:
        1. Get top-k candidates using vector search
        2. Rerank and return top-n results
        """
        # Stage 1: Vector search
        results = self.collection.query(
            query_texts=[query],
            n_results=initial_k,
            include=["documents", "metadatas", "distances"]
        )
        
        candidates = results["documents"][0]
        metadatas = results["metadatas"][0]
        initial_distances = results["distances"][0]
        
        if not candidates:
            return []
        
        # Stage 2: Rerank with cross-encoder
        pairs = [[query, doc] for doc in candidates]
        rerank_scores = self.reranker.predict(pairs)
        
        # Combine results with scores
        ranked_results = []
        for i, (doc, metadata, initial_dist, rerank_score) in enumerate(
            zip(candidates, metadatas, initial_distances, rerank_scores)
        ):
            ranked_results.append({
                "document": doc,
                "metadata": metadata,
                "initial_distance": initial_dist,
                "rerank_score": float(rerank_score),
                "initial_rank": i + 1
            })
        
        # Sort by rerank score (higher is better)
        ranked_results.sort(key=lambda x: x["rerank_score"], reverse=True)
        
        return ranked_results[:final_k]
    
    def compare_rankings(self, query: str, k: int = 5) -> None:
        """Compare initial vs reranked results"""
        results = self.retrieve(query, initial_k=k, final_k=k)
        
        print(f"\n🔍 Query: {query}")
        print("=" * 70)
        print(f"{'Rerank':<8} {'Initial':<8} {'Score':<10} {'Document':<40}")
        print("-" * 70)
        
        for new_rank, r in enumerate(results, 1):
            doc_preview = r["document"][:40] + "..."
            print(f"{new_rank:<8} {r['initial_rank']:<8} {r['rerank_score']:.4f}    {doc_preview}")

# Example usage
print("\n🎖️ Reranking RAG Example:")
print("=" * 60)

rerank_rag = RerankerRAG(
    collection_name="demo_rerank",
    persist_directory="./demo_rerank"
)
rerank_rag.add_documents(sample_documents)

# Compare rankings
rerank_rag.compare_rankings("How do neural networks learn from data?")

In [ ]:
# ============================================
# 🎖️ COHERE RERANKING (Production-Grade)
# ============================================

class CohereRerankerRAG:
    """
    RAG with Cohere's production-grade reranking API
    Cohere's reranker is more powerful than local cross-encoders
    """
    
    def __init__(
        self,
        collection_name: str = "cohere_rerank",
        persist_directory: str = "./chroma_cohere"
    ):
        self.collection_name = collection_name
        
        # Initialize ChromaDB
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name="all-MiniLM-L6-v2"
        )
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            embedding_function=self.embedding_function
        )
        
        # Initialize Cohere client
        import cohere
        self.cohere_client = cohere.Client(os.environ.get("COHERE_API_KEY", ""))
        
        print("✅ CohereRerankerRAG initialized")
    
    def add_documents(self, documents: List[Document]) -> None:
        """Add documents to collection"""
        texts = [doc.page_content for doc in documents]
        metadatas = [doc.metadata for doc in documents]
        ids = [f"doc_{i}" for i in range(len(documents))]
        
        self.collection.add(documents=texts, metadatas=metadatas, ids=ids)
        print(f"📥 Added {len(documents)} documents")
    
    def retrieve_and_rerank(
        self,
        query: str,
        initial_k: int = 20,
        final_k: int = 5,
        model: str = "rerank-english-v3.0"
    ) -> List[Dict[str, Any]]:
        """
        Retrieve with ChromaDB, rerank with Cohere
        """
        # Stage 1: Vector retrieval
        results = self.collection.query(
            query_texts=[query],
            n_results=initial_k,
            include=["documents", "metadatas"]
        )
        
        documents = results["documents"][0]
        metadatas = results["metadatas"][0]
        
        if not documents:
            return []
        
        # Stage 2: Cohere reranking
        rerank_results = self.cohere_client.rerank(
            model=model,
            query=query,
            documents=documents,
            top_n=final_k,
            return_documents=True
        )
        
        # Format results
        final_results = []
        for r in rerank_results.results:
            final_results.append({
                "document": r.document.text,
                "relevance_score": r.relevance_score,
                "original_index": r.index,
                "metadata": metadatas[r.index]
            })
        
        return final_results

print("\n💡 CohereRerankerRAG class defined!")
print("   Requires COHERE_API_KEY environment variable")

---
## 5. 🔀 Hybrid Search RAG (Dense + Sparse)

Combines semantic search (dense vectors) with keyword search (sparse/BM25) for the best of both worlds.

**Benefits:**
- Dense: Captures semantic meaning
- Sparse: Catches exact keyword matches
- Together: Better recall and precision

In [ ]:
# ============================================
# 🔀 HYBRID SEARCH RAG (Dense + Sparse)
# ============================================

from rank_bm25 import BM25Okapi
import numpy as np

class HybridRAG:
    """
    Hybrid search combining:
    - Dense retrieval (semantic embeddings via ChromaDB)
    - Sparse retrieval (BM25 keyword matching)
    
    Uses Reciprocal Rank Fusion (RRF) to combine results
    """
    
    def __init__(
        self,
        collection_name: str = "hybrid_rag",
        embedding_model: str = "all-MiniLM-L6-v2",
        persist_directory: str = "./chroma_hybrid",
        alpha: float = 0.5  # Weight for dense vs sparse (0.5 = equal)
    ):
        self.collection_name = collection_name
        self.alpha = alpha
        self.documents = []
        self.tokenized_docs = []
        self.bm25 = None
        
        # Initialize ChromaDB for dense search
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name=embedding_model
        )
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            embedding_function=self.embedding_function
        )
        
        print(f"✅ HybridRAG initialized (alpha={alpha})")
    
    def _tokenize(self, text: str) -> List[str]:
        """Simple tokenization for BM25"""
        return text.lower().split()
    
    def add_documents(self, documents: List[Document]) -> None:
        """Add documents to both dense and sparse indices"""
        self.documents = documents
        texts = [doc.page_content for doc in documents]
        metadatas = [doc.metadata for doc in documents]
        ids = [f"doc_{i}" for i in range(len(documents))]
        
        # Add to ChromaDB (dense)
        self.collection.add(
            documents=texts,
            metadatas=metadatas,
            ids=ids
        )
        
        # Build BM25 index (sparse)
        self.tokenized_docs = [self._tokenize(text) for text in texts]
        self.bm25 = BM25Okapi(self.tokenized_docs)
        
        print(f"📥 Added {len(documents)} documents to hybrid index")
    
    def _dense_search(self, query: str, k: int) -> List[Tuple[int, float]]:
        """Dense vector search using ChromaDB"""
        results = self.collection.query(
            query_texts=[query],
            n_results=k,
            include=["documents", "distances"]
        )
        
        # Convert distances to scores (lower distance = higher score)
        scores = []
        for i, dist in enumerate(results["distances"][0]):
            # Cosine distance to similarity
            score = 1 - dist
            scores.append((i, score))
        
        return scores
    
    def _sparse_search(self, query: str, k: int) -> List[Tuple[int, float]]:
        """Sparse BM25 search"""
        tokenized_query = self._tokenize(query)
        bm25_scores = self.bm25.get_scores(tokenized_query)
        
        # Get top-k indices
        top_indices = np.argsort(bm25_scores)[::-1][:k]
        scores = [(int(idx), float(bm25_scores[idx])) for idx in top_indices]
        
        return scores
    
    def _reciprocal_rank_fusion(
        self,
        dense_results: List[Tuple[int, float]],
        sparse_results: List[Tuple[int, float]],
        k: int = 60
    ) -> List[Tuple[int, float]]:
        """
        Reciprocal Rank Fusion (RRF) to combine rankings
        
        RRF score = sum(1 / (k + rank)) for each list
        """
        rrf_scores = {}
        
        # Score from dense results
        for rank, (doc_idx, _) in enumerate(dense_results):
            if doc_idx not in rrf_scores:
                rrf_scores[doc_idx] = 0
            rrf_scores[doc_idx] += self.alpha * (1 / (k + rank + 1))
        
        # Score from sparse results
        for rank, (doc_idx, _) in enumerate(sparse_results):
            if doc_idx not in rrf_scores:
                rrf_scores[doc_idx] = 0
            rrf_scores[doc_idx] += (1 - self.alpha) * (1 / (k + rank + 1))
        
        # Sort by RRF score
        sorted_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
        return sorted_results
    
    def search(
        self,
        query: str,
        k: int = 5,
        return_scores: bool = True
    ) -> List[Dict[str, Any]]:
        """
        Hybrid search combining dense and sparse retrieval
        """
        # Get results from both methods
        dense_results = self._dense_search(query, k * 2)
        sparse_results = self._sparse_search(query, k * 2)
        
        # Combine with RRF
        combined = self._reciprocal_rank_fusion(dense_results, sparse_results)[:k]
        
        # Format results
        results = []
        for doc_idx, rrf_score in combined:
            results.append({
                "document": self.documents[doc_idx].page_content,
                "metadata": self.documents[doc_idx].metadata,
                "rrf_score": rrf_score,
                "doc_index": doc_idx
            })
        
        return results
    
    def compare_methods(self, query: str, k: int = 3) -> None:
        """Compare dense, sparse, and hybrid results"""
        print(f"\n🔍 Query: {query}")
        print("=" * 70)
        
        # Dense only
        dense = self._dense_search(query, k)
        print("\n📊 Dense Search (Semantic):")
        for rank, (idx, score) in enumerate(dense, 1):
            doc = self.documents[idx].page_content[:50]
            print(f"   {rank}. [{score:.4f}] {doc}...")
        
        # Sparse only
        sparse = self._sparse_search(query, k)
        print("\n📊 Sparse Search (BM25):")
        for rank, (idx, score) in enumerate(sparse, 1):
            doc = self.documents[idx].page_content[:50]
            print(f"   {rank}. [{score:.4f}] {doc}...")
        
        # Hybrid
        hybrid = self.search(query, k)
        print("\n📊 Hybrid Search (RRF):")
        for rank, result in enumerate(hybrid, 1):
            doc = result["document"][:50]
            print(f"   {rank}. [{result['rrf_score']:.4f}] {doc}...")

# Example usage
print("\n🔀 Hybrid Search RAG Example:")
print("=" * 60)

hybrid_rag = HybridRAG(
    collection_name="demo_hybrid",
    persist_directory="./demo_hybrid",
    alpha=0.5
)
hybrid_rag.add_documents(sample_documents)

# Compare search methods
hybrid_rag.compare_methods("neural network deep learning layers")

---
## 6. 🔄 Multi-Query RAG

Generate multiple query variations to improve retrieval recall.

**Approaches:**
- Query expansion/rewriting
- Hypothetical Document Embeddings (HyDE)
- Step-back prompting

In [ ]:
# ============================================
# 🔄 MULTI-QUERY RAG
# ============================================

class MultiQueryRAG:
    """
    Generate multiple query variations to improve retrieval recall.
    
    Techniques:
    1. Query expansion - Generate similar queries
    2. HyDE - Generate hypothetical document, then search
    3. Step-back - Generate broader, more abstract queries
    """
    
    def __init__(
        self,
        collection_name: str = "multiquery_rag",
        persist_directory: str = "./chroma_multiquery",
        llm_client=None
    ):
        self.collection_name = collection_name
        self.llm_client = llm_client
        
        # Initialize ChromaDB
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name="all-MiniLM-L6-v2"
        )
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            embedding_function=self.embedding_function
        )
        
        print("✅ MultiQueryRAG initialized")
    
    def add_documents(self, documents: List[Document]) -> None:
        """Add documents to collection"""
        texts = [doc.page_content for doc in documents]
        metadatas = [doc.metadata for doc in documents]
        ids = [f"doc_{i}" for i in range(len(documents))]
        
        self.collection.add(documents=texts, metadatas=metadatas, ids=ids)
        print(f"📥 Added {len(documents)} documents")
    
    def generate_query_variations(
        self,
        query: str,
        num_variations: int = 3
    ) -> List[str]:
        """
        Generate multiple query variations using an LLM
        Falls back to simple variations if no LLM available
        """
        if self.llm_client is None:
            # Simple rule-based variations
            variations = [
                query,
                f"What is {query}?",
                f"Explain {query}",
                f"Tell me about {query}",
            ]
            return variations[:num_variations + 1]
        
        prompt = f"""Generate {num_variations} different ways to ask the following question.
Each variation should capture the same intent but use different words.

Original question: {query}

Return only the variations, one per line, without numbering."""

        response = self.llm_client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}]
        )
        
        variations = response.choices[0].message.content.strip().split('\n')
        return [query] + [v.strip() for v in variations if v.strip()]
    
    def generate_hypothetical_document(self, query: str) -> str:
        """
        HyDE: Generate a hypothetical document that would answer the query
        Then use this to search (better semantic matching)
        """
        if self.llm_client is None:
            return query  # Fallback
        
        prompt = f"""Write a short paragraph that would be a perfect answer to the following question.
Write as if you are the source document, not answering the question.

Question: {query}

Document:"""

        response = self.llm_client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}]
        )
        
        return response.choices[0].message.content.strip()
    
    def generate_stepback_query(self, query: str) -> str:
        """
        Step-back prompting: Generate a more abstract/general query
        Helps retrieve broader context
        """
        if self.llm_client is None:
            return query
        
        prompt = f"""Given the following specific question, generate a more general, abstract question
that would help understand the broader context.

Specific question: {query}

General question:"""

        response = self.llm_client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}]
        )
        
        return response.choices[0].message.content.strip()
    
    def multi_query_search(
        self,
        query: str,
        k_per_query: int = 3,
        final_k: int = 5,
        method: str = "expansion"  # "expansion", "hyde", "stepback", "all"
    ) -> List[Dict[str, Any]]:
        """
        Search using multiple query variations
        """
        queries = [query]
        
        if method in ["expansion", "all"]:
            variations = self.generate_query_variations(query, 3)
            queries.extend(variations)
        
        if method in ["hyde", "all"]:
            hyde_doc = self.generate_hypothetical_document(query)
            queries.append(hyde_doc)
        
        if method in ["stepback", "all"]:
            stepback = self.generate_stepback_query(query)
            queries.append(stepback)
        
        # Remove duplicates while preserving order
        seen = set()
        unique_queries = []
        for q in queries:
            if q.lower() not in seen:
                seen.add(q.lower())
                unique_queries.append(q)
        
        print(f"🔄 Searching with {len(unique_queries)} query variations")
        
        # Search with all queries
        all_results = {}
        for q in unique_queries:
            results = self.collection.query(
                query_texts=[q],
                n_results=k_per_query,
                include=["documents", "metadatas", "distances"]
            )
            
            for doc, meta, dist in zip(
                results["documents"][0],
                results["metadatas"][0],
                results["distances"][0]
            ):
                doc_key = doc[:100]  # Use first 100 chars as key
                if doc_key not in all_results:
                    all_results[doc_key] = {
                        "document": doc,
                        "metadata": meta,
                        "best_distance": dist,
                        "query_count": 1
                    }
                else:
                    all_results[doc_key]["query_count"] += 1
                    all_results[doc_key]["best_distance"] = min(
                        all_results[doc_key]["best_distance"], dist
                    )
        
        # Score by combination of distance and query count
        scored_results = []
        for r in all_results.values():
            # Higher score = better (more queries matched, lower distance)
            score = r["query_count"] * (1 - r["best_distance"])
            scored_results.append({
                **r,
                "combined_score": score
            })
        
        # Sort by combined score
        scored_results.sort(key=lambda x: x["combined_score"], reverse=True)
        
        return scored_results[:final_k]

# Example usage
print("\n🔄 Multi-Query RAG Example:")
print("=" * 60)

mq_rag = MultiQueryRAG(
    collection_name="demo_multiquery",
    persist_directory="./demo_multiquery"
)
mq_rag.add_documents(sample_documents)

# Generate query variations (without LLM)
query = "How do transformers work?"
variations = mq_rag.generate_query_variations(query)
print(f"\n📝 Query variations for: '{query}'")
for i, v in enumerate(variations, 1):
    print(f"   {i}. {v}")

# Multi-query search
results = mq_rag.multi_query_search(query, method="expansion")
print(f"\n🔍 Found {len(results)} results")
for i, r in enumerate(results, 1):
    print(f"\n   [{i}] Score: {r['combined_score']:.4f} (matched {r['query_count']} queries)")
    print(f"       {r['document'][:60]}...")

---
## 7. 🔍 Self-Query RAG (Metadata Filtering)

Automatically extract metadata filters from natural language queries.

**Example:**
- Query: "Find documents about NLP from chapter 3"
- Extracts: `topic="nlp"`, `chapter=3`

In [ ]:
# ============================================
# 🔍 SELF-QUERY RAG (Metadata Filtering)
# ============================================

import re

class SelfQueryRAG:
    """
    Automatically extract metadata filters from natural language queries.
    
    Combines semantic search with structured metadata filtering.
    """
    
    def __init__(
        self,
        collection_name: str = "selfquery_rag",
        persist_directory: str = "./chroma_selfquery",
        metadata_fields: Dict[str, Dict] = None,
        llm_client=None
    ):
        self.collection_name = collection_name
        self.llm_client = llm_client
        
        # Define metadata schema
        self.metadata_fields = metadata_fields or {
            "topic": {
                "type": "string",
                "description": "The topic of the document",
                "values": ["machine_learning", "deep_learning", "nlp", "rag", 
                          "vector_databases", "transformers", "fine_tuning", 
                          "prompt_engineering"]
            },
            "chapter": {
                "type": "integer",
                "description": "The chapter number"
            },
            "source": {
                "type": "string",
                "description": "The source file name"
            }
        }
        
        # Initialize ChromaDB
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name="all-MiniLM-L6-v2"
        )
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            embedding_function=self.embedding_function
        )
        
        print("✅ SelfQueryRAG initialized")
    
    def add_documents(self, documents: List[Document]) -> None:
        """Add documents with metadata"""
        texts = [doc.page_content for doc in documents]
        metadatas = [doc.metadata for doc in documents]
        ids = [f"doc_{i}" for i in range(len(documents))]
        
        self.collection.add(documents=texts, metadatas=metadatas, ids=ids)
        print(f"📥 Added {len(documents)} documents with metadata")
    
    def _parse_query_with_llm(self, query: str) -> Tuple[str, Dict]:
        """Use LLM to extract filters from query"""
        schema_desc = "\n".join([
            f"- {k}: {v['description']} (type: {v['type']})"
            for k, v in self.metadata_fields.items()
        ])
        
        prompt = f"""Given a user query, extract:
1. The semantic search query (what to search for)
2. Any metadata filters

Metadata fields available:
{schema_desc}

User query: {query}

Return in this exact format:
SEARCH: <semantic query>
FILTERS: <key=value pairs, comma-separated, or "none">

Example:
User query: "Find NLP documents from chapter 3"
SEARCH: NLP natural language processing
FILTERS: topic=nlp, chapter=3"""

        if self.llm_client is None:
            # Rule-based fallback
            return self._parse_query_rules(query)
        
        response = self.llm_client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}]
        )
        
        content = response.choices[0].message.content
        return self._parse_llm_response(content)
    
    def _parse_query_rules(self, query: str) -> Tuple[str, Dict]:
        """
        Rule-based query parsing (fallback when no LLM)
        """
        filters = {}
        clean_query = query.lower()
        
        # Extract chapter numbers
        chapter_match = re.search(r'chapter\s*(\d+)', clean_query)
        if chapter_match:
            filters["chapter"] = int(chapter_match.group(1))
            clean_query = re.sub(r'chapter\s*\d+', '', clean_query)
        
        # Extract known topics
        topic_keywords = {
            "machine learning": "machine_learning",
            "ml": "machine_learning",
            "deep learning": "deep_learning",
            "dl": "deep_learning",
            "nlp": "nlp",
            "natural language": "nlp",
            "rag": "rag",
            "retrieval": "rag",
            "vector": "vector_databases",
            "transformer": "transformers",
            "fine-tuning": "fine_tuning",
            "finetuning": "fine_tuning",
            "prompt": "prompt_engineering"
        }
        
        for keyword, topic in topic_keywords.items():
            if keyword in clean_query:
                filters["topic"] = topic
                break
        
        # Remove filter keywords from query
        for keyword in ["about", "from", "in", "on", "find", "search", "documents"]:
            clean_query = clean_query.replace(keyword, "")
        
        return clean_query.strip(), filters
    
    def _parse_llm_response(self, response: str) -> Tuple[str, Dict]:
        """Parse LLM response into query and filters"""
        lines = response.strip().split('\n')
        search_query = ""
        filters = {}
        
        for line in lines:
            if line.startswith("SEARCH:"):
                search_query = line.replace("SEARCH:", "").strip()
            elif line.startswith("FILTERS:"):
                filter_str = line.replace("FILTERS:", "").strip()
                if filter_str.lower() != "none":
                    for pair in filter_str.split(","):
                        if "=" in pair:
                            key, value = pair.split("=", 1)
                            key = key.strip()
                            value = value.strip()
                            # Convert to appropriate type
                            if key in self.metadata_fields:
                                if self.metadata_fields[key]["type"] == "integer":
                                    value = int(value)
                            filters[key] = value
        
        return search_query or response, filters
    
    def _build_chroma_filter(self, filters: Dict) -> Optional[Dict]:
        """Build ChromaDB-compatible filter"""
        if not filters:
            return None
        
        if len(filters) == 1:
            key, value = list(filters.items())[0]
            return {key: {"$eq": value}}
        
        # Multiple filters: use $and
        conditions = [{key: {"$eq": value}} for key, value in filters.items()]
        return {"$and": conditions}
    
    def search(
        self,
        query: str,
        k: int = 5,
        auto_filter: bool = True
    ) -> Dict[str, Any]:
        """
        Search with automatic metadata filter extraction
        """
        if auto_filter:
            search_query, filters = self._parse_query_rules(query)
        else:
            search_query = query
            filters = {}
        
        chroma_filter = self._build_chroma_filter(filters)
        
        results = self.collection.query(
            query_texts=[search_query],
            n_results=k,
            where=chroma_filter,
            include=["documents", "metadatas", "distances"]
        )
        
        return {
            "original_query": query,
            "search_query": search_query,
            "filters": filters,
            "documents": results["documents"][0],
            "metadatas": results["metadatas"][0],
            "distances": results["distances"][0]
        }

# Example usage
print("\n🔍 Self-Query RAG Example:")
print("=" * 60)

sq_rag = SelfQueryRAG(
    collection_name="demo_selfquery",
    persist_directory="./demo_selfquery"
)
sq_rag.add_documents(sample_documents)

# Test queries with automatic filter extraction
test_queries = [
    "Find documents about NLP",
    "What is deep learning from chapter 2",
    "Tell me about RAG systems",
    "Machine learning basics from chapter 1"
]

for query in test_queries:
    result = sq_rag.search(query)
    print(f"\n📝 Query: '{query}'")
    print(f"   → Search: '{result['search_query']}'")
    print(f"   → Filters: {result['filters']}")
    print(f"   → Found: {len(result['documents'])} documents")

---
## 8. 📄 Parent Document Retriever

Store small chunks for precise retrieval, but return larger parent documents for better context.

**Strategy:**
1. Split documents into small chunks
2. Store chunks with reference to parent
3. Retrieve chunks, but return parent documents

In [ ]:
# ============================================
# 📄 PARENT DOCUMENT RETRIEVER
# ============================================

class ParentDocumentRAG:
    """
    Two-tier retrieval system:
    - Store small chunks for precise matching
    - Return larger parent documents for context
    
    This gives you the best of both worlds:
    - Precise retrieval (small chunks match better)
    - Rich context (full documents provide complete information)
    """
    
    def __init__(
        self,
        collection_name: str = "parent_doc_rag",
        persist_directory: str = "./chroma_parent",
        child_chunk_size: int = 200,
        child_chunk_overlap: int = 50
    ):
        self.collection_name = collection_name
        self.child_chunk_size = child_chunk_size
        self.child_chunk_overlap = child_chunk_overlap
        
        # Store for parent documents
        self.parent_docs: Dict[str, Document] = {}
        
        # Initialize ChromaDB for child chunks
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name="all-MiniLM-L6-v2"
        )
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            embedding_function=self.embedding_function
        )
        
        # Text splitter for creating child chunks
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=child_chunk_size,
            chunk_overlap=child_chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""]
        )
        
        print("✅ ParentDocumentRAG initialized")
    
    def add_documents(self, documents: List[Document]) -> None:
        """
        Add documents:
        1. Store parent documents in memory/docstore
        2. Split into child chunks
        3. Store child chunks in ChromaDB with parent reference
        """
        all_chunks = []
        all_metadatas = []
        all_ids = []
        
        for i, doc in enumerate(documents):
            parent_id = f"parent_{i}"
            
            # Store parent document
            self.parent_docs[parent_id] = doc
            
            # Create child chunks
            chunks = self.splitter.split_text(doc.page_content)
            
            for j, chunk in enumerate(chunks):
                child_id = f"{parent_id}_child_{j}"
                
                # Store chunk with reference to parent
                metadata = doc.metadata.copy()
                metadata["parent_id"] = parent_id
                metadata["chunk_index"] = j
                metadata["total_chunks"] = len(chunks)
                
                all_chunks.append(chunk)
                all_metadatas.append(metadata)
                all_ids.append(child_id)
        
        # Add all chunks to ChromaDB
        self.collection.add(
            documents=all_chunks,
            metadatas=all_metadatas,
            ids=all_ids
        )
        
        print(f"📥 Added {len(documents)} parent documents")
        print(f"   → Created {len(all_chunks)} child chunks")
    
    def retrieve_chunks(
        self,
        query: str,
        k: int = 5
    ) -> List[Dict[str, Any]]:
        """Retrieve matching child chunks"""
        results = self.collection.query(
            query_texts=[query],
            n_results=k,
            include=["documents", "metadatas", "distances"]
        )
        
        chunks = []
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        ):
            chunks.append({
                "chunk": doc,
                "metadata": meta,
                "distance": dist,
                "parent_id": meta["parent_id"]
            })
        
        return chunks
    
    def retrieve_with_parents(
        self,
        query: str,
        k_chunks: int = 5,
        return_unique_parents: bool = True
    ) -> Dict[str, Any]:
        """
        Retrieve child chunks and return their parent documents
        """
        # Get matching chunks
        chunks = self.retrieve_chunks(query, k_chunks)
        
        # Get unique parent documents
        parent_ids = []
        seen = set()
        for chunk in chunks:
            pid = chunk["parent_id"]
            if pid not in seen:
                parent_ids.append(pid)
                seen.add(pid)
        
        # Fetch parent documents
        parents = []
        for pid in parent_ids:
            if pid in self.parent_docs:
                parents.append({
                    "parent_id": pid,
                    "document": self.parent_docs[pid],
                    "matched_chunks": [
                        c for c in chunks if c["parent_id"] == pid
                    ]
                })
        
        return {
            "query": query,
            "matched_chunks": chunks,
            "parent_documents": parents
        }
    
    def retrieve_with_context_window(
        self,
        query: str,
        k_chunks: int = 3,
        context_window: int = 1
    ) -> List[Dict[str, Any]]:
        """
        Retrieve chunks with surrounding context
        (neighboring chunks from the same parent)
        """
        chunks = self.retrieve_chunks(query, k_chunks)
        
        expanded_results = []
        for chunk in chunks:
            parent_id = chunk["parent_id"]
            chunk_idx = chunk["metadata"]["chunk_index"]
            total_chunks = chunk["metadata"]["total_chunks"]
            
            # Get surrounding chunk indices
            start_idx = max(0, chunk_idx - context_window)
            end_idx = min(total_chunks, chunk_idx + context_window + 1)
            
            # Fetch surrounding chunks from collection
            neighbor_ids = [
                f"{parent_id}_child_{i}" 
                for i in range(start_idx, end_idx)
            ]
            
            neighbors = self.collection.get(
                ids=neighbor_ids,
                include=["documents", "metadatas"]
            )
            
            expanded_results.append({
                "matched_chunk": chunk,
                "context_chunks": list(zip(
                    neighbors["ids"],
                    neighbors["documents"],
                    neighbors["metadatas"]
                ))
            })
        
        return expanded_results

# Example usage
print("\n📄 Parent Document Retriever Example:")
print("=" * 60)

parent_rag = ParentDocumentRAG(
    collection_name="demo_parent",
    persist_directory="./demo_parent",
    child_chunk_size=150,
    child_chunk_overlap=30
)
parent_rag.add_documents(sample_documents)

# Query and compare chunk vs parent retrieval
query = "What is the transformer attention mechanism?"
print(f"\n🔍 Query: {query}")

# Get chunks only
chunks = parent_rag.retrieve_chunks(query, k=3)
print(f"\n📝 Retrieved Chunks ({len(chunks)}):")
for i, c in enumerate(chunks, 1):
    print(f"   {i}. [{c['distance']:.3f}] {c['chunk'][:60]}...")

# Get with parents
results = parent_rag.retrieve_with_parents(query)
print(f"\n📚 Parent Documents ({len(results['parent_documents'])}):")
for p in results['parent_documents']:
    doc = p['document'].page_content[:80]
    print(f"   - {p['parent_id']}: {doc}...")

---
## 9. 🤖 Multi-Agent RAG System

Coordinate multiple specialized agents for complex RAG tasks.

**Agents:**
- 🔍 **Retriever Agent** - Fetches relevant documents
- 🧠 **Analyzer Agent** - Analyzes and synthesizes information
- ✅ **Validator Agent** - Checks for hallucinations
- 📝 **Response Agent** - Generates final response

In [ ]:
# ============================================
# 🤖 MULTI-AGENT RAG SYSTEM
# ============================================

from abc import ABC, abstractmethod
from enum import Enum
from dataclasses import dataclass, field

class AgentRole(Enum):
    RETRIEVER = "retriever"
    ANALYZER = "analyzer"
    VALIDATOR = "validator"
    RESPONDER = "responder"
    ROUTER = "router"

@dataclass
class AgentMessage:
    """Message passed between agents"""
    sender: AgentRole
    content: Any
    metadata: Dict[str, Any] = field(default_factory=dict)

class BaseAgent(ABC):
    """Base class for all RAG agents"""
    
    def __init__(self, role: AgentRole, llm_client=None):
        self.role = role
        self.llm_client = llm_client
    
    @abstractmethod
    def process(self, message: AgentMessage) -> AgentMessage:
        pass
    
    def _call_llm(self, prompt: str) -> str:
        """Call LLM with prompt"""
        if self.llm_client is None:
            return f"[LLM Response Placeholder for: {prompt[:100]}...]"
        
        response = self.llm_client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content


class RetrieverAgent(BaseAgent):
    """Agent responsible for document retrieval"""
    
    def __init__(self, vector_store, llm_client=None):
        super().__init__(AgentRole.RETRIEVER, llm_client)
        self.vector_store = vector_store
    
    def process(self, message: AgentMessage) -> AgentMessage:
        query = message.content
        k = message.metadata.get("k", 5)
        
        # Retrieve documents
        results = self.vector_store.query(
            query_texts=[query],
            n_results=k,
            include=["documents", "metadatas", "distances"]
        )
        
        documents = []
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        ):
            documents.append({
                "content": doc,
                "metadata": meta,
                "relevance_score": 1 - dist
            })
        
        return AgentMessage(
            sender=self.role,
            content=documents,
            metadata={"query": query, "count": len(documents)}
        )


class AnalyzerAgent(BaseAgent):
    """Agent that analyzes and synthesizes retrieved documents"""
    
    def __init__(self, llm_client=None):
        super().__init__(AgentRole.ANALYZER, llm_client)
    
    def process(self, message: AgentMessage) -> AgentMessage:
        documents = message.content
        query = message.metadata.get("query", "")
        
        # Create analysis prompt
        doc_texts = "\n\n---\n\n".join([
            f"Document {i+1} (relevance: {d['relevance_score']:.2f}):\n{d['content']}"
            for i, d in enumerate(documents)
        ])
        
        prompt = f"""Analyze the following documents to answer the query.
Extract key information, identify patterns, and note any contradictions.

Query: {query}

Documents:
{doc_texts}

Analysis:
1. Key findings:
2. Relevant information:
3. Any gaps or contradictions:
4. Confidence level (high/medium/low):"""

        analysis = self._call_llm(prompt)
        
        return AgentMessage(
            sender=self.role,
            content={
                "analysis": analysis,
                "documents": documents,
                "query": query
            },
            metadata={"analyzed_docs": len(documents)}
        )


class ValidatorAgent(BaseAgent):
    """Agent that validates responses against source documents"""
    
    def __init__(self, llm_client=None):
        super().__init__(AgentRole.VALIDATOR, llm_client)
    
    def process(self, message: AgentMessage) -> AgentMessage:
        response = message.content.get("response", "")
        documents = message.content.get("documents", [])
        
        # Create validation prompt
        doc_texts = "\n\n".join([d["content"] for d in documents])
        
        prompt = f"""Validate if the following response is supported by the source documents.
Check for:
1. Factual accuracy
2. Potential hallucinations
3. Missing important information

Response to validate:
{response}

Source documents:
{doc_texts}

Validation result:
- Is accurate: (yes/no)
- Hallucinations found: (list any)
- Missing info: (list any)
- Suggested corrections: (if any)"""

        validation = self._call_llm(prompt)
        
        return AgentMessage(
            sender=self.role,
            content={
                "validation": validation,
                "original_response": response,
                "is_validated": True
            },
            metadata={}
        )


class ResponderAgent(BaseAgent):
    """Agent that generates final response"""
    
    def __init__(self, llm_client=None):
        super().__init__(AgentRole.RESPONDER, llm_client)
    
    def process(self, message: AgentMessage) -> AgentMessage:
        analysis = message.content.get("analysis", "")
        documents = message.content.get("documents", [])
        query = message.content.get("query", "")
        
        # Create response prompt
        prompt = f"""Based on the analysis and source documents, generate a comprehensive response.

Query: {query}

Analysis:
{analysis}

Requirements:
1. Be accurate and cite sources when possible
2. Be concise but complete
3. Acknowledge any uncertainties
4. Use clear, professional language

Response:"""

        response = self._call_llm(prompt)
        
        return AgentMessage(
            sender=self.role,
            content={
                "response": response,
                "documents": documents,
                "query": query
            },
            metadata={"response_length": len(response)}
        )


class RouterAgent(BaseAgent):
    """Agent that routes queries to appropriate pipelines"""
    
    def __init__(self, llm_client=None):
        super().__init__(AgentRole.ROUTER, llm_client)
        
        self.query_types = {
            "factual": "Direct fact lookup - use simple retrieval",
            "analytical": "Requires analysis - use multi-step retrieval",
            "comparative": "Comparing concepts - use broader retrieval",
            "creative": "Needs synthesis - use comprehensive pipeline"
        }
    
    def process(self, message: AgentMessage) -> AgentMessage:
        query = message.content
        
        # Simple rule-based routing (can be enhanced with LLM)
        query_lower = query.lower()
        
        if any(word in query_lower for word in ["what is", "define", "who is"]):
            query_type = "factual"
        elif any(word in query_lower for word in ["compare", "difference", "vs"]):
            query_type = "comparative"
        elif any(word in query_lower for word in ["why", "how does", "explain"]):
            query_type = "analytical"
        else:
            query_type = "factual"
        
        return AgentMessage(
            sender=self.role,
            content={
                "query": query,
                "query_type": query_type,
                "pipeline": self.query_types[query_type]
            },
            metadata={"routed_to": query_type}
        )


class MultiAgentRAG:
    """
    Orchestrates multiple agents for complex RAG tasks
    """
    
    def __init__(
        self,
        collection_name: str = "multiagent_rag",
        persist_directory: str = "./chroma_multiagent",
        llm_client=None
    ):
        # Initialize ChromaDB
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name="all-MiniLM-L6-v2"
        )
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            embedding_function=self.embedding_function
        )
        
        # Initialize agents
        self.router = RouterAgent(llm_client)
        self.retriever = RetrieverAgent(self.collection, llm_client)
        self.analyzer = AnalyzerAgent(llm_client)
        self.validator = ValidatorAgent(llm_client)
        self.responder = ResponderAgent(llm_client)
        
        print("✅ MultiAgentRAG initialized with 5 agents")
    
    def add_documents(self, documents: List[Document]) -> None:
        """Add documents to the collection"""
        texts = [doc.page_content for doc in documents]
        metadatas = [doc.metadata for doc in documents]
        ids = [f"doc_{i}" for i in range(len(documents))]
        
        self.collection.add(documents=texts, metadatas=metadatas, ids=ids)
        print(f"📥 Added {len(documents)} documents")
    
    def query(
        self,
        query: str,
        validate: bool = True,
        verbose: bool = True
    ) -> Dict[str, Any]:
        """
        Process query through multi-agent pipeline
        """
        messages = []
        
        # Step 1: Route the query
        if verbose:
            print(f"\n🔀 [Router] Analyzing query type...")
        route_msg = self.router.process(AgentMessage(
            sender=AgentRole.ROUTER,
            content=query
        ))
        messages.append(("router", route_msg))
        if verbose:
            print(f"   → Query type: {route_msg.content['query_type']}")
        
        # Step 2: Retrieve documents
        if verbose:
            print(f"\n🔍 [Retriever] Fetching relevant documents...")
        retrieve_msg = self.retriever.process(AgentMessage(
            sender=AgentRole.ROUTER,
            content=query,
            metadata={"k": 5}
        ))
        messages.append(("retriever", retrieve_msg))
        if verbose:
            print(f"   → Retrieved {retrieve_msg.metadata['count']} documents")
        
        # Step 3: Analyze documents
        if verbose:
            print(f"\n🧠 [Analyzer] Analyzing documents...")
        analyze_msg = self.analyzer.process(AgentMessage(
            sender=AgentRole.RETRIEVER,
            content=retrieve_msg.content,
            metadata={"query": query}
        ))
        messages.append(("analyzer", analyze_msg))
        if verbose:
            print(f"   → Analysis complete")
        
        # Step 4: Generate response
        if verbose:
            print(f"\n📝 [Responder] Generating response...")
        response_msg = self.responder.process(AgentMessage(
            sender=AgentRole.ANALYZER,
            content=analyze_msg.content
        ))
        messages.append(("responder", response_msg))
        
        # Step 5: Validate (optional)
        if validate:
            if verbose:
                print(f"\n✅ [Validator] Validating response...")
            validate_msg = self.validator.process(AgentMessage(
                sender=AgentRole.RESPONDER,
                content=response_msg.content
            ))
            messages.append(("validator", validate_msg))
            if verbose:
                print(f"   → Validation complete")
        
        return {
            "query": query,
            "query_type": route_msg.content["query_type"],
            "response": response_msg.content["response"],
            "documents": retrieve_msg.content,
            "analysis": analyze_msg.content["analysis"],
            "validation": validate_msg.content if validate else None,
            "agent_messages": messages
        }

# Example usage
print("\n🤖 Multi-Agent RAG Example:")
print("=" * 60)

ma_rag = MultiAgentRAG(
    collection_name="demo_multiagent",
    persist_directory="./demo_multiagent"
)
ma_rag.add_documents(sample_documents)

# Process a query
result = ma_rag.query("How do transformers use attention mechanisms?", verbose=True)

print("\n" + "=" * 60)
print("📋 FINAL RESULT:")
print(f"Query Type: {result['query_type']}")
print(f"Documents Retrieved: {len(result['documents'])}")
print(f"\nResponse:\n{result['response'][:300]}...")

---
## 10. 📊 Corrective RAG (CRAG)

Self-correcting RAG that evaluates retrieval quality and takes corrective actions.

**Flow:**
1. Retrieve documents
2. Grade relevance
3. If relevant → Generate
4. If not relevant → Web search / Reformulate query

In [ ]:
# ============================================
# 📊 CORRECTIVE RAG (CRAG)
# ============================================

class CorrectiveRAG:
    """
    Self-correcting RAG that evaluates retrieval quality and takes corrective actions.
    
    Inspired by the CRAG paper (Corrective Retrieval Augmented Generation)
    
    Flow:
    1. Retrieve documents
    2. Grade each document's relevance
    3. Based on grades:
       - All relevant → Use documents
       - Partially relevant → Refine and supplement
       - Not relevant → Reformulate query or use web search
    """
    
    def __init__(
        self,
        collection_name: str = "crag",
        persist_directory: str = "./chroma_crag",
        relevance_threshold: float = 0.7,
        llm_client=None
    ):
        self.relevance_threshold = relevance_threshold
        self.llm_client = llm_client
        
        # Initialize ChromaDB
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name="all-MiniLM-L6-v2"
        )
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            embedding_function=self.embedding_function
        )
        
        # Initialize reranker for grading
        from sentence_transformers import CrossEncoder
        self.grader = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        
        print("✅ CorrectiveRAG initialized")
    
    def add_documents(self, documents: List[Document]) -> None:
        """Add documents to collection"""
        texts = [doc.page_content for doc in documents]
        metadatas = [doc.metadata for doc in documents]
        ids = [f"doc_{i}" for i in range(len(documents))]
        
        self.collection.add(documents=texts, metadatas=metadatas, ids=ids)
        print(f"📥 Added {len(documents)} documents")
    
    def retrieve(self, query: str, k: int = 5) -> List[Dict]:
        """Initial retrieval"""
        results = self.collection.query(
            query_texts=[query],
            n_results=k,
            include=["documents", "metadatas", "distances"]
        )
        
        documents = []
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        ):
            documents.append({
                "content": doc,
                "metadata": meta,
                "distance": dist
            })
        
        return documents
    
    def grade_documents(
        self,
        query: str,
        documents: List[Dict]
    ) -> List[Dict]:
        """
        Grade each document's relevance to the query
        Returns documents with relevance scores
        """
        graded_docs = []
        
        for doc in documents:
            # Score relevance using cross-encoder
            score = self.grader.predict([[query, doc["content"]]])[0]
            
            graded_docs.append({
                **doc,
                "relevance_score": float(score),
                "is_relevant": float(score) >= self.relevance_threshold
            })
        
        return graded_docs
    
    def _reformulate_query(self, query: str) -> str:
        """Reformulate query to improve retrieval"""
        if self.llm_client is None:
            # Simple keyword expansion
            return f"{query} definition explanation overview"
        
        prompt = f"""The following query didn't retrieve good results.
Reformulate it to be more specific and likely to match relevant documents.

Original query: {query}

Reformulated query:"""
        
        response = self.llm_client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content.strip()
    
    def _refine_document(self, query: str, document: str) -> str:
        """Extract only the relevant parts of a document"""
        if self.llm_client is None:
            return document
        
        prompt = f"""Extract only the parts of this document that are relevant to the query.
Remove irrelevant information.

Query: {query}

Document:
{document}

Relevant excerpt:"""
        
        response = self.llm_client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content.strip()
    
    def corrective_retrieve(
        self,
        query: str,
        k: int = 5,
        max_retries: int = 2
    ) -> Dict[str, Any]:
        """
        Main CRAG pipeline with corrective actions
        """
        action_log = []
        
        # Step 1: Initial retrieval
        documents = self.retrieve(query, k)
        action_log.append(("retrieve", f"Retrieved {len(documents)} documents"))
        
        # Step 2: Grade documents
        graded_docs = self.grade_documents(query, documents)
        action_log.append(("grade", f"Graded {len(graded_docs)} documents"))
        
        relevant = [d for d in graded_docs if d["is_relevant"]]
        ambiguous = [d for d in graded_docs if 0.3 <= d["relevance_score"] < self.relevance_threshold]
        
        relevance_ratio = len(relevant) / len(graded_docs) if graded_docs else 0
        
        # Step 3: Take corrective action based on grading
        final_documents = []
        
        if relevance_ratio >= 0.6:
            # CORRECT: Most documents are relevant
            action_log.append(("action", "CORRECT - Using relevant documents"))
            final_documents = relevant
            
        elif relevance_ratio >= 0.2:
            # AMBIGUOUS: Some relevant, some not
            action_log.append(("action", "AMBIGUOUS - Refining and supplementing"))
            
            # Refine ambiguous documents
            for doc in ambiguous:
                refined = self._refine_document(query, doc["content"])
                final_documents.append({**doc, "content": refined, "refined": True})
            
            # Add clearly relevant ones
            final_documents.extend(relevant)
            
        else:
            # INCORRECT: Need to reformulate query
            action_log.append(("action", "INCORRECT - Reformulating query"))
            
            for retry in range(max_retries):
                new_query = self._reformulate_query(query)
                action_log.append(("reformulate", f"Retry {retry+1}: '{new_query}'"))
                
                new_docs = self.retrieve(new_query, k)
                new_graded = self.grade_documents(query, new_docs)  # Grade against original
                new_relevant = [d for d in new_graded if d["is_relevant"]]
                
                if new_relevant:
                    final_documents = new_relevant
                    break
            
            # If still nothing, use best available
            if not final_documents:
                action_log.append(("fallback", "Using best available documents"))
                graded_docs.sort(key=lambda x: x["relevance_score"], reverse=True)
                final_documents = graded_docs[:3]
        
        return {
            "query": query,
            "documents": final_documents,
            "action_log": action_log,
            "relevance_ratio": relevance_ratio,
            "total_retrieved": len(documents),
            "relevant_count": len(relevant)
        }

# Example usage
print("\n📊 Corrective RAG (CRAG) Example:")
print("=" * 60)

crag = CorrectiveRAG(
    collection_name="demo_crag",
    persist_directory="./demo_crag",
    relevance_threshold=0.5
)
crag.add_documents(sample_documents)

# Test with different queries
test_queries = [
    "What is machine learning?",  # Should find good matches
    "How do transformer models work?",  # Good match
    "What is quantum computing?",  # Poor match - needs correction
]

for query in test_queries:
    result = crag.corrective_retrieve(query)
    print(f"\n🔍 Query: '{query}'")
    print(f"   Relevance: {result['relevance_ratio']:.1%}")
    print(f"   Actions taken:")
    for action, detail in result['action_log']:
        print(f"      [{action}] {detail}")

---
## 11. 💬 Conversational RAG with Memory

RAG with conversation history for multi-turn interactions.

**Features:**
- Maintains chat history
- Contextualizes follow-up questions
- Manages context window limits

In [ ]:
# ============================================
# 💬 CONVERSATIONAL RAG WITH MEMORY
# ============================================

from collections import deque

@dataclass
class ChatMessage:
    role: str  # "user" or "assistant"
    content: str
    timestamp: str = None
    retrieved_docs: List[str] = None

class ConversationalRAG:
    """
    RAG with conversation memory for multi-turn interactions.
    
    Features:
    - Maintains conversation history
    - Contextualizes follow-up questions
    - Manages context window limits
    - Retrieves relevant documents based on conversation context
    """
    
    def __init__(
        self,
        collection_name: str = "conversational_rag",
        persist_directory: str = "./chroma_conv",
        max_history: int = 10,
        llm_client=None
    ):
        self.max_history = max_history
        self.llm_client = llm_client
        self.conversation_history: deque = deque(maxlen=max_history)
        
        # Initialize ChromaDB
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name="all-MiniLM-L6-v2"
        )
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            embedding_function=self.embedding_function
        )
        
        print("✅ ConversationalRAG initialized")
    
    def add_documents(self, documents: List[Document]) -> None:
        """Add documents to collection"""
        texts = [doc.page_content for doc in documents]
        metadatas = [doc.metadata for doc in documents]
        ids = [f"doc_{i}" for i in range(len(documents))]
        
        self.collection.add(documents=texts, metadatas=metadatas, ids=ids)
        print(f"📥 Added {len(documents)} documents")
    
    def _get_conversation_context(self) -> str:
        """Format conversation history for context"""
        if not self.conversation_history:
            return ""
        
        context_parts = []
        for msg in self.conversation_history:
            role = "Human" if msg.role == "user" else "Assistant"
            context_parts.append(f"{role}: {msg.content}")
        
        return "\n".join(context_parts)
    
    def _contextualize_query(self, query: str) -> str:
        """
        Reformulate the query based on conversation history.
        Makes follow-up questions standalone.
        """
        if not self.conversation_history:
            return query
        
        history = self._get_conversation_context()
        
        if self.llm_client is None:
            # Simple heuristic: if query contains pronouns, add context
            pronouns = ["it", "they", "this", "that", "these", "those", "he", "she"]
            if any(p in query.lower().split() for p in pronouns):
                # Get last topic from history
                last_msg = self.conversation_history[-1]
                return f"{query} (context: {last_msg.content[:100]})"
            return query
        
        prompt = f"""Given the conversation history and a follow-up question, 
reformulate the question to be standalone and include all necessary context.

Conversation history:
{history}

Follow-up question: {query}

Standalone question:"""
        
        response = self.llm_client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content.strip()
    
    def retrieve(self, query: str, k: int = 5) -> List[Dict]:
        """Retrieve documents based on contextualized query"""
        # Contextualize the query
        contextualized_query = self._contextualize_query(query)
        
        results = self.collection.query(
            query_texts=[contextualized_query],
            n_results=k,
            include=["documents", "metadatas", "distances"]
        )
        
        documents = []
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        ):
            documents.append({
                "content": doc,
                "metadata": meta,
                "distance": dist
            })
        
        return documents
    
    def _generate_response(
        self,
        query: str,
        documents: List[Dict],
        use_history: bool = True
    ) -> str:
        """Generate response using context and history"""
        # Build context from documents
        doc_context = "\n\n".join([
            f"Document {i+1}:\n{d['content']}"
            for i, d in enumerate(documents)
        ])
        
        # Build conversation history
        history_context = ""
        if use_history and self.conversation_history:
            history_context = f"\nConversation history:\n{self._get_conversation_context()}\n"
        
        prompt = f"""You are a helpful assistant. Answer the question based on the context provided.
If you reference previous conversation, do so naturally.

{history_context}

Relevant documents:
{doc_context}

Current question: {query}

Answer:"""
        
        if self.llm_client is None:
            return f"[Response based on {len(documents)} documents for: {query}]"
        
        response = self.llm_client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content
    
    def chat(self, query: str, k: int = 5) -> Dict[str, Any]:
        """
        Main chat interface - handles the full conversational RAG flow
        """
        from datetime import datetime
        
        # Retrieve relevant documents
        documents = self.retrieve(query, k)
        
        # Generate response
        response = self._generate_response(query, documents)
        
        # Add to conversation history
        self.conversation_history.append(ChatMessage(
            role="user",
            content=query,
            timestamp=datetime.now().isoformat()
        ))
        
        self.conversation_history.append(ChatMessage(
            role="assistant",
            content=response,
            timestamp=datetime.now().isoformat(),
            retrieved_docs=[d["content"][:100] for d in documents]
        ))
        
        return {
            "query": query,
            "response": response,
            "documents": documents,
            "history_length": len(self.conversation_history)
        }
    
    def clear_history(self) -> None:
        """Clear conversation history"""
        self.conversation_history.clear()
        print("🗑️ Conversation history cleared")
    
    def get_history(self) -> List[Dict]:
        """Get formatted conversation history"""
        return [
            {"role": msg.role, "content": msg.content}
            for msg in self.conversation_history
        ]

# Example usage
print("\n💬 Conversational RAG Example:")
print("=" * 60)

conv_rag = ConversationalRAG(
    collection_name="demo_conv",
    persist_directory="./demo_conv"
)
conv_rag.add_documents(sample_documents)

# Simulate a conversation
conversation = [
    "What is machine learning?",
    "How is deep learning related to it?",  # Follow-up
    "What are transformers?",
    "How do they use attention?",  # Follow-up
]

for query in conversation:
    result = conv_rag.chat(query)
    print(f"\n👤 User: {query}")
    print(f"🤖 Assistant: {result['response'][:150]}...")
    print(f"   (Retrieved {len(result['documents'])} docs, History: {result['history_length']} messages)")

---
## 12. 📈 RAG Evaluation Metrics

Essential metrics to evaluate RAG system performance.

**Retrieval Metrics:**
- Precision, Recall, MRR, NDCG

**Generation Metrics:**
- Faithfulness, Answer Relevancy, Context Relevancy

In [ ]:
# ============================================
# 📈 RAG EVALUATION METRICS
# ============================================

import numpy as np
from typing import Set

class RAGEvaluator:
    """
    Comprehensive evaluation metrics for RAG systems.
    
    Covers:
    1. Retrieval Quality Metrics
    2. Generation Quality Metrics
    3. End-to-End Metrics
    """
    
    def __init__(self, llm_client=None):
        self.llm_client = llm_client
    
    # ==========================================
    # RETRIEVAL METRICS
    # ==========================================
    
    @staticmethod
    def precision_at_k(
        retrieved_ids: List[str],
        relevant_ids: Set[str],
        k: int = None
    ) -> float:
        """
        Precision@K: Fraction of retrieved documents that are relevant
        
        Formula: |Retrieved ∩ Relevant| / |Retrieved|
        """
        if k:
            retrieved_ids = retrieved_ids[:k]
        
        relevant_retrieved = sum(1 for doc_id in retrieved_ids if doc_id in relevant_ids)
        return relevant_retrieved / len(retrieved_ids) if retrieved_ids else 0.0
    
    @staticmethod
    def recall_at_k(
        retrieved_ids: List[str],
        relevant_ids: Set[str],
        k: int = None
    ) -> float:
        """
        Recall@K: Fraction of relevant documents that were retrieved
        
        Formula: |Retrieved ∩ Relevant| / |Relevant|
        """
        if k:
            retrieved_ids = retrieved_ids[:k]
        
        relevant_retrieved = sum(1 for doc_id in retrieved_ids if doc_id in relevant_ids)
        return relevant_retrieved / len(relevant_ids) if relevant_ids else 0.0
    
    @staticmethod
    def f1_at_k(
        retrieved_ids: List[str],
        relevant_ids: Set[str],
        k: int = None
    ) -> float:
        """
        F1@K: Harmonic mean of Precision and Recall
        """
        precision = RAGEvaluator.precision_at_k(retrieved_ids, relevant_ids, k)
        recall = RAGEvaluator.recall_at_k(retrieved_ids, relevant_ids, k)
        
        if precision + recall == 0:
            return 0.0
        return 2 * (precision * recall) / (precision + recall)
    
    @staticmethod
    def mrr(
        retrieved_ids: List[str],
        relevant_ids: Set[str]
    ) -> float:
        """
        Mean Reciprocal Rank: 1/position of first relevant document
        
        Higher is better (1.0 = first result is relevant)
        """
        for i, doc_id in enumerate(retrieved_ids, 1):
            if doc_id in relevant_ids:
                return 1.0 / i
        return 0.0
    
    @staticmethod
    def ndcg_at_k(
        retrieved_ids: List[str],
        relevant_ids: Set[str],
        k: int = 10
    ) -> float:
        """
        Normalized Discounted Cumulative Gain
        
        Accounts for position of relevant documents (earlier is better)
        """
        retrieved_ids = retrieved_ids[:k]
        
        # Calculate DCG
        dcg = 0.0
        for i, doc_id in enumerate(retrieved_ids, 1):
            rel = 1 if doc_id in relevant_ids else 0
            dcg += rel / np.log2(i + 1)
        
        # Calculate Ideal DCG
        ideal_relevances = [1] * min(len(relevant_ids), k)
        ideal_relevances += [0] * (k - len(ideal_relevances))
        
        idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal_relevances))
        
        return dcg / idcg if idcg > 0 else 0.0
    
    @staticmethod
    def hit_rate(
        retrieved_ids: List[str],
        relevant_ids: Set[str]
    ) -> float:
        """
        Hit Rate: Whether any relevant document was retrieved
        
        Returns 1.0 if at least one relevant doc found, 0.0 otherwise
        """
        return 1.0 if any(doc_id in relevant_ids for doc_id in retrieved_ids) else 0.0
    
    # ==========================================
    # GENERATION METRICS (LLM-based)
    # ==========================================
    
    def faithfulness(
        self,
        response: str,
        context: List[str]
    ) -> float:
        """
        Faithfulness: Is the response grounded in the context?
        
        Checks if claims in response are supported by retrieved documents.
        Returns score between 0 and 1.
        """
        if self.llm_client is None:
            return 0.5  # Placeholder
        
        context_text = "\n\n".join(context)
        
        prompt = f"""Evaluate if the response is faithful to the given context.
A faithful response only contains information that can be verified from the context.

Context:
{context_text}

Response:
{response}

Score the faithfulness from 0.0 to 1.0 where:
- 1.0 = Completely faithful, all claims supported by context
- 0.5 = Partially faithful, some unsupported claims
- 0.0 = Unfaithful, mostly unsupported or contradicting claims

Return only the numeric score:"""
        
        result = self.llm_client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}]
        )
        
        try:
            score = float(result.choices[0].message.content.strip())
            return min(max(score, 0.0), 1.0)
        except:
            return 0.5
    
    def answer_relevancy(
        self,
        query: str,
        response: str
    ) -> float:
        """
        Answer Relevancy: Does the response actually answer the question?
        """
        if self.llm_client is None:
            return 0.5  # Placeholder
        
        prompt = f"""Evaluate if the response directly and completely answers the question.

Question: {query}

Response: {response}

Score the answer relevancy from 0.0 to 1.0 where:
- 1.0 = Directly and completely answers the question
- 0.5 = Partially answers or tangentially related
- 0.0 = Does not answer the question at all

Return only the numeric score:"""
        
        result = self.llm_client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": prompt}]
        )
        
        try:
            score = float(result.choices[0].message.content.strip())
            return min(max(score, 0.0), 1.0)
        except:
            return 0.5
    
    def context_relevancy(
        self,
        query: str,
        context: List[str]
    ) -> float:
        """
        Context Relevancy: Are the retrieved documents relevant to the query?
        
        Measures retrieval quality from content perspective.
        """
        if self.llm_client is None:
            return 0.5  # Placeholder
        
        # Evaluate each context piece
        scores = []
        for ctx in context:
            prompt = f"""Is this document relevant to the query?

Query: {query}

Document: {ctx[:500]}

Score relevancy from 0.0 to 1.0:
- 1.0 = Highly relevant, directly addresses the query
- 0.5 = Somewhat relevant, contains related information
- 0.0 = Not relevant at all

Return only the numeric score:"""
            
            result = self.llm_client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": prompt}]
            )
            
            try:
                score = float(result.choices[0].message.content.strip())
                scores.append(min(max(score, 0.0), 1.0))
            except:
                scores.append(0.5)
        
        return np.mean(scores) if scores else 0.0
    
    # ==========================================
    # COMPREHENSIVE EVALUATION
    # ==========================================
    
    def evaluate_retrieval(
        self,
        retrieved_ids: List[str],
        relevant_ids: Set[str],
        k: int = 5
    ) -> Dict[str, float]:
        """Compute all retrieval metrics"""
        return {
            "precision@k": self.precision_at_k(retrieved_ids, relevant_ids, k),
            "recall@k": self.recall_at_k(retrieved_ids, relevant_ids, k),
            "f1@k": self.f1_at_k(retrieved_ids, relevant_ids, k),
            "mrr": self.mrr(retrieved_ids, relevant_ids),
            "ndcg@k": self.ndcg_at_k(retrieved_ids, relevant_ids, k),
            "hit_rate": self.hit_rate(retrieved_ids, relevant_ids),
        }
    
    def evaluate_generation(
        self,
        query: str,
        response: str,
        context: List[str]
    ) -> Dict[str, float]:
        """Compute all generation metrics"""
        return {
            "faithfulness": self.faithfulness(response, context),
            "answer_relevancy": self.answer_relevancy(query, response),
            "context_relevancy": self.context_relevancy(query, context),
        }

# Example usage
print("\n📈 RAG Evaluation Example:")
print("=" * 60)

evaluator = RAGEvaluator()

# Simulated retrieval results
retrieved = ["doc_1", "doc_3", "doc_5", "doc_2", "doc_7"]
relevant = {"doc_1", "doc_2", "doc_4"}

# Evaluate retrieval
retrieval_metrics = evaluator.evaluate_retrieval(retrieved, relevant, k=5)
print("\n📊 Retrieval Metrics:")
for metric, value in retrieval_metrics.items():
    print(f"   {metric}: {value:.4f}")

# Simulated generation evaluation
print("\n📊 Generation Metrics (require LLM):")
print("   faithfulness: [requires LLM client]")
print("   answer_relevancy: [requires LLM client]")
print("   context_relevancy: [requires LLM client]")

---
## 13. 🛠️ Production-Ready RAG with LangChain

Complete production-ready RAG implementation using LangChain with ChromaDB.

In [ ]:
# ============================================
# 🛠️ PRODUCTION-READY RAG WITH LANGCHAIN
# ============================================

class ProductionRAG:
    """
    Production-ready RAG implementation with LangChain.
    
    Features:
    - Multiple embedding options
    - Configurable retrieval strategies
    - Response streaming
    - Error handling
    - Logging and monitoring
    """
    
    def __init__(
        self,
        collection_name: str = "production_rag",
        persist_directory: str = "./chroma_production",
        embedding_type: str = "huggingface",  # "openai", "huggingface", "cohere"
        embedding_model: str = "all-MiniLM-L6-v2",
        llm_model: str = "gpt-3.5-turbo",
        chunk_size: int = 500,
        chunk_overlap: int = 50
    ):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        
        # Initialize embeddings
        self.embeddings = self._init_embeddings(embedding_type, embedding_model)
        
        # Initialize text splitter
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""]
        )
        
        # Initialize vector store
        self.vectorstore = Chroma(
            collection_name=collection_name,
            embedding_function=self.embeddings,
            persist_directory=persist_directory
        )
        
        # Initialize LLM (if available)
        try:
            self.llm = ChatOpenAI(model_name=llm_model, temperature=0)
        except:
            self.llm = None
            print("⚠️ LLM not initialized (missing API key)")
        
        print(f"✅ ProductionRAG initialized")
        print(f"   Embedding: {embedding_type} ({embedding_model})")
        print(f"   Vector Store: ChromaDB ({collection_name})")
    
    def _init_embeddings(self, embedding_type: str, model_name: str):
        """Initialize embedding model based on type"""
        if embedding_type == "openai":
            return OpenAIEmbeddings(model=model_name)
        elif embedding_type == "huggingface":
            return HuggingFaceEmbeddings(model_name=model_name)
        elif embedding_type == "cohere":
            return CohereEmbeddings(model=model_name)
        else:
            raise ValueError(f"Unknown embedding type: {embedding_type}")
    
    def add_documents(
        self,
        documents: List[Document],
        batch_size: int = 100
    ) -> None:
        """Add documents with chunking and batching"""
        # Split documents into chunks
        all_chunks = []
        for doc in documents:
            chunks = self.text_splitter.split_documents([doc])
            all_chunks.extend(chunks)
        
        # Add in batches
        for i in range(0, len(all_chunks), batch_size):
            batch = all_chunks[i:i + batch_size]
            self.vectorstore.add_documents(batch)
            print(f"   Added batch {i//batch_size + 1}/{(len(all_chunks)-1)//batch_size + 1}")
        
        print(f"📥 Added {len(documents)} documents as {len(all_chunks)} chunks")
    
    def add_texts(
        self,
        texts: List[str],
        metadatas: List[Dict] = None
    ) -> None:
        """Add raw texts"""
        self.vectorstore.add_texts(texts, metadatas)
        print(f"📥 Added {len(texts)} texts")
    
    def similarity_search(
        self,
        query: str,
        k: int = 5,
        filter: Dict = None
    ) -> List[Document]:
        """Basic similarity search"""
        return self.vectorstore.similarity_search(
            query,
            k=k,
            filter=filter
        )
    
    def similarity_search_with_score(
        self,
        query: str,
        k: int = 5
    ) -> List[Tuple[Document, float]]:
        """Similarity search with relevance scores"""
        return self.vectorstore.similarity_search_with_score(query, k=k)
    
    def mmr_search(
        self,
        query: str,
        k: int = 5,
        fetch_k: int = 20,
        lambda_mult: float = 0.5
    ) -> List[Document]:
        """
        Maximum Marginal Relevance search.
        Balances relevance with diversity.
        """
        return self.vectorstore.max_marginal_relevance_search(
            query,
            k=k,
            fetch_k=fetch_k,
            lambda_mult=lambda_mult
        )
    
    def get_retriever(
        self,
        search_type: str = "similarity",
        k: int = 5,
        **kwargs
    ):
        """Get a LangChain retriever"""
        search_kwargs = {"k": k, **kwargs}
        
        return self.vectorstore.as_retriever(
            search_type=search_type,
            search_kwargs=search_kwargs
        )
    
    def create_qa_chain(
        self,
        chain_type: str = "stuff",
        retriever_k: int = 5,
        custom_prompt: PromptTemplate = None
    ):
        """Create a RetrievalQA chain"""
        if self.llm is None:
            raise ValueError("LLM not initialized. Set OPENAI_API_KEY.")
        
        retriever = self.get_retriever(k=retriever_k)
        
        if custom_prompt:
            return RetrievalQA.from_chain_type(
                llm=self.llm,
                chain_type=chain_type,
                retriever=retriever,
                chain_type_kwargs={"prompt": custom_prompt},
                return_source_documents=True
            )
        
        return RetrievalQA.from_chain_type(
            llm=self.llm,
            chain_type=chain_type,
            retriever=retriever,
            return_source_documents=True
        )
    
    def query(
        self,
        question: str,
        k: int = 5,
        return_sources: bool = True
    ) -> Dict[str, Any]:
        """
        Complete RAG query with source documents
        """
        # Retrieve
        docs_with_scores = self.similarity_search_with_score(question, k)
        docs = [doc for doc, _ in docs_with_scores]
        scores = [score for _, score in docs_with_scores]
        
        # Build context
        context = "\n\n---\n\n".join([doc.page_content for doc in docs])
        
        # Generate response
        if self.llm:
            prompt = f"""Use the following context to answer the question.
If you can't answer based on the context, say "I don't have enough information."

Context:
{context}

Question: {question}

Answer:"""
            
            response = self.llm.invoke(prompt)
            answer = response.content
        else:
            answer = "[LLM not available - configure OPENAI_API_KEY]"
        
        result = {
            "question": question,
            "answer": answer,
        }
        
        if return_sources:
            result["sources"] = [
                {
                    "content": doc.page_content[:200] + "...",
                    "metadata": doc.metadata,
                    "score": float(score)
                }
                for doc, score in zip(docs, scores)
            ]
        
        return result

# Example usage
print("\n🛠️ Production RAG Example:")
print("=" * 60)

prod_rag = ProductionRAG(
    collection_name="demo_production",
    persist_directory="./demo_production",
    embedding_type="huggingface",
    embedding_model="all-MiniLM-L6-v2"
)

prod_rag.add_documents(sample_documents)

# Query
result = prod_rag.query("What is the relationship between machine learning and deep learning?")
print(f"\n❓ Question: {result['question']}")
print(f"\n💬 Answer: {result['answer'][:200]}...")
print(f"\n📚 Sources: {len(result.get('sources', []))} documents")

---
## 14. 🔥 Quick Reference: RAG Patterns Cheatsheet

In [ ]:
# ============================================
# 🔥 QUICK REFERENCE: RAG PATTERNS CHEATSHEET
# ============================================

RAG_PATTERNS = """
╔══════════════════════════════════════════════════════════════════════════════╗
║                         🔥 RAG PATTERNS CHEATSHEET                           ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  1️⃣  SIMPLE RAG                                                              ║
║      Query → Embed → Vector Search → Top-K Docs → LLM → Answer               ║
║      Best for: Simple Q&A, small knowledge bases                             ║
║                                                                              ║
║  2️⃣  CHUNKED RAG                                                             ║
║      Docs → Chunk → Embed → Store | Query → Search → Retrieve → Generate    ║
║      Chunking options: Fixed, Recursive, Semantic, Token-based               ║
║      Best for: Long documents, PDFs, articles                                ║
║                                                                              ║
║  3️⃣  RERANKING RAG                                                           ║
║      Query → Fast Search (Top-N) → Rerank (Cross-Encoder) → Top-K → LLM      ║
║      Best for: Precision-critical applications                               ║
║                                                                              ║
║  4️⃣  HYBRID RAG (Dense + Sparse)                                             ║
║      Query → [Vector Search] + [BM25] → RRF Fusion → Final Results           ║
║      Best for: Technical docs, code, exact term matching needed              ║
║                                                                              ║
║  5️⃣  MULTI-QUERY RAG                                                         ║
║      Query → Generate Variations → Search Each → Dedupe & Merge → LLM        ║
║      Variations: Expansion, HyDE, Step-back                                  ║
║      Best for: Complex queries, ambiguous questions                          ║
║                                                                              ║
║  6️⃣  SELF-QUERY RAG                                                          ║
║      Query → Extract Filters + Semantic Query → Filtered Search → LLM        ║
║      Best for: Structured data, metadata-rich documents                      ║
║                                                                              ║
║  7️⃣  PARENT DOCUMENT RETRIEVER                                               ║
║      Docs → Split to Chunks → Index Chunks | Query → Match Chunks → Return   ║
║                                              Parent Docs                     ║
║      Best for: Needing full context, legal docs, research papers             ║
║                                                                              ║
║  8️⃣  MULTI-AGENT RAG                                                         ║
║      Query → Router → Retriever → Analyzer → Generator → Validator → Answer  ║
║      Best for: Complex workflows, enterprise applications                    ║
║                                                                              ║
║  9️⃣  CORRECTIVE RAG (CRAG)                                                   ║
║      Query → Retrieve → Grade Docs → [Correct/Ambiguous/Incorrect]           ║
║                                    → Corrective Action → Generate            ║
║      Best for: High-stakes applications, reducing hallucinations             ║
║                                                                              ║
║  🔟 CONVERSATIONAL RAG                                                       ║
║      Query + History → Contextualize → Retrieve → Generate → Update History  ║
║      Best for: Chatbots, multi-turn conversations                            ║
║                                                                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                           📊 WHEN TO USE WHAT                                ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  ┌─────────────────────────┬────────────────────────────────────────────┐    ║
║  │ Use Case                │ Recommended Pattern                        │    ║
║  ├─────────────────────────┼────────────────────────────────────────────┤    ║
║  │ Simple FAQ bot          │ Simple RAG                                 │    ║
║  │ Document search         │ Chunked RAG + Reranking                    │    ║
║  │ Code search             │ Hybrid RAG (BM25 helps with code)          │    ║
║  │ Legal/Medical docs      │ Parent Document + CRAG                     │    ║
║  │ Customer support        │ Conversational RAG + Multi-Agent           │    ║
║  │ Research assistant      │ Multi-Query + Reranking                    │    ║
║  │ Enterprise search       │ Self-Query + Hybrid + Reranking            │    ║
║  │ High-stakes decisions   │ CRAG + Multi-Agent + Validation            │    ║
║  └─────────────────────────┴────────────────────────────────────────────┘    ║
║                                                                              ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                          ⚡ OPTIMIZATION TIPS                                ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  CHUNKING:                                                                   ║
║  • Smaller chunks (200-500) = Better precision, less context                 ║
║  • Larger chunks (500-1000) = More context, might dilute relevance           ║
║  • Use overlap (10-20%) to avoid losing context at boundaries                ║
║  • Match chunk size to your embedding model's optimal input length           ║
║                                                                              ║
║  RETRIEVAL:                                                                  ║
║  • Start with k=5-10, adjust based on context window limits                  ║
║  • Use reranking for top-k > 3 results                                       ║
║  • MMR for diversity when answers might come from different sections         ║
║  • Hybrid search when exact matches matter (names, codes, IDs)               ║
║                                                                              ║
║  EMBEDDINGS:                                                                 ║
║  • all-MiniLM-L6-v2: Fast, good for general use                              ║
║  • text-embedding-3-small: Better quality, requires API                      ║
║  • BGE models: Great for multilingual                                        ║
║  • Domain-specific: Fine-tune for specialized vocabularies                   ║
║                                                                              ║
║  PROMPTING:                                                                  ║
║  • Include "based on the context" to reduce hallucinations                   ║
║  • Ask model to cite sources when possible                                   ║
║  • Use system prompts to set response format                                 ║
║  • Chain-of-thought for complex reasoning                                    ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

print(RAG_PATTERNS)

---
## 15. 🧹 Cleanup Utility

Remove demo ChromaDB directories created during this tutorial.

In [ ]:
# ============================================
# 🧹 CLEANUP UTILITY
# ============================================

import shutil
from pathlib import Path

def cleanup_demo_directories():
    """Remove all demo ChromaDB directories created during this tutorial"""
    demo_dirs = [
        "./demo_chroma",
        "./demo_fixed",
        "./demo_recursive", 
        "./demo_rerank",
        "./demo_hybrid",
        "./demo_multiquery",
        "./demo_selfquery",
        "./demo_parent",
        "./demo_multiagent",
        "./demo_crag",
        "./demo_conv",
        "./demo_production",
        "./chroma_db",
        "./chroma_chunked",
        "./chroma_rerank",
        "./chroma_cohere",
        "./chroma_hybrid",
        "./chroma_multiquery",
        "./chroma_selfquery",
        "./chroma_parent",
        "./chroma_multiagent",
        "./chroma_crag",
        "./chroma_conv",
        "./chroma_production",
    ]
    
    removed = 0
    for dir_path in demo_dirs:
        path = Path(dir_path)
        if path.exists():
            shutil.rmtree(path)
            removed += 1
            print(f"   🗑️ Removed: {dir_path}")
    
    if removed > 0:
        print(f"\n✅ Cleaned up {removed} demo directories")
    else:
        print("✨ No demo directories to clean up")

# Uncomment to run cleanup:
# cleanup_demo_directories()

---
## 📚 Resources & References

### Documentation
- [ChromaDB Docs](https://docs.trychroma.com/)
- [LangChain RAG](https://python.langchain.com/docs/use_cases/question_answering/)
- [Sentence Transformers](https://www.sbert.net/)

### Papers
- **RAG**: [Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks](https://arxiv.org/abs/2005.11401)
- **CRAG**: [Corrective Retrieval Augmented Generation](https://arxiv.org/abs/2401.15884)
- **HyDE**: [Precise Zero-Shot Dense Retrieval without Relevance Labels](https://arxiv.org/abs/2212.10496)
- **Self-RAG**: [Learning to Retrieve, Generate, and Critique](https://arxiv.org/abs/2310.11511)

### Key Libraries
```python
# Core
pip install chromadb langchain langchain-openai langchain-community

# Embeddings
pip install sentence-transformers huggingface-hub

# Reranking
pip install cohere  # or use sentence-transformers cross-encoders

# Hybrid Search  
pip install rank_bm25

# Evaluation
pip install ragas  # RAG evaluation framework
```

---
## 🎉 Happy RAG Building!

This cheatsheet covers the most important RAG patterns and implementations.
Mix and match these patterns based on your specific use case!

**Created with ❤️ using ChromaDB and Python**